# RespiScan Pipeline: Preprocessing & Data Extraction
Notebook ini digunakan untuk mengolah data awal suara batuk, melakukan standardisasi durasi audio (1 detik), melakukan Subject-Level Splitting untuk mencegah data leakage (kebocoran pembicara), mengekstrak fitur akustik (MFCC & Continuous Wavelet Transform Scalogram), melakukan pencampuran noise lingkungan seimbang (class-balanced noise mixing) pada training set, serta menerapkan augmentasi akustik kelas Healthy (pitch/speed shift) untuk menyeimbangkan kelas.

### 1. Inisialisasi & Konfigurasi Global
Mengimpor pustaka utama, mengatur Windows kernel crash bypass, serta menginisialisasi CUDA GPU device.

In [ ]:
import os
import gc
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Mencegah window pop-up di Windows
import matplotlib.pyplot as plt
import pywt
import torch
import torchaudio
import torchaudio.transforms as T
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Windows crash kernel fix
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Hardware Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Perangkat tempur aktif: {DEVICE}")

### 2. Konfigurasi Path Global
Mendefinisikan lokasi file metadata, folder output, serta folder kebisingan lingkungan (*noise database*).

In [ ]:
# --- PATH KONFIGURASI ---
PATH_METADATA = r'Metadata.csv'
PATH_OUTPUT_RAW = r'RespiScan_Final_Dataset'
PATH_OUTPUT_FINAL = r'RespiScan_Train_Ready'
PATH_AUDIO_NTB = r'dataset_tbc'

BASE_DIRS = {
    "Nairobi_Old": r'dataset_tbc'
}

NOISE_DIRS = [
    r'DKITCHEN',
    r'NPARK',
    r'OHALLWAY'
]

# Pembuatan folder output
os.makedirs(PATH_OUTPUT_RAW, exist_ok=True)
os.makedirs(PATH_OUTPUT_FINAL, exist_ok=True)

print("✅ Semua konfigurasi path berhasil diinisialisasi secara rapi.")

### 3. Pemetaan & Audit Metadata Awal
Memetakan subjek pasien dari Metadata.csv ke file audio fisik dan menganalisis sebaran kelas.

In [ ]:
# 1. Load referensi metadata
if os.path.exists(PATH_METADATA):
    df_meta = pd.read_csv(PATH_METADATA)
    df_meta.columns = df_meta.columns.str.strip()
    df_meta['subject'] = df_meta['subject'].astype(str).str.strip()
    
    def get_clean_subject_id(subj):
        import re
        return re.sub(r'_now_C$', '', subj)
    
    df_meta['clean_subject'] = df_meta['subject'].apply(get_clean_subject_id)
    
    # Scan available wav files in dataset_tbc
    DIR_AUDIO = r"dataset_tbc"
    wav_files = [f for f in os.listdir(DIR_AUDIO) if f.lower().endswith('.wav')]
    print(f"Total .wav files in {DIR_AUDIO}: {len(wav_files)}")
    
    # Map wav files to clean subject IDs
    subject_to_files = {subj: [] for subj in df_meta['clean_subject'].unique()}
    for f in wav_files:
        for clean_subj in sorted(subject_to_files.keys(), key=len, reverse=True):
            if f.startswith(clean_subj):
                subject_to_files[clean_subj].append(f)
                break
                
    df_meta['file_count'] = df_meta['clean_subject'].map(lambda x: len(subject_to_files.get(x, [])))
    df_available = df_meta[df_meta['file_count'] > 0].copy()
    print(f"Subjects with audio files: {len(df_available)}")
    print(df_available['Label'].value_counts())
else:
    print("⚠️ File Metadata.csv tidak ditemukan di direktori lokal!")

### 4. Standardisasi Sinyal Audio & Pencampuran Noise
Mendefinisikan fungsi modular untuk standardisasi channel mono, resampling ke 44.1 kHz, pemotongan/padding durasi 1 detik, serta pencampuran noise lingkungan.

In [ ]:
def standardize_audio(waveform, sr, target_sr=16000, target_samples=16000):
    """Menstandardisasi audio ke format 1-channel mono, 16.0 kHz, dan panjang tepat 1 detik."""
    # 1. Pastikan Mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # 2. Resampling ke 16.0 kHz
    if sr != target_sr:
        resampler = T.Resample(sr, target_sr)
        waveform = resampler(waveform)
    
    # 3. Potong/Padding durasi tepat 1 detik
    if waveform.shape[1] > target_samples:
        waveform = waveform[:, :target_samples]
    else:
        padding = target_samples - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, padding))
        
    return waveform

def get_random_noise_segment(dirs):
    """Memotong segmen acak berdurasi 1 detik dari database kebisingan."""
    selected_dir = random.choice(dirs)
    if not os.path.exists(selected_dir):
        return torch.zeros(1, 16000)
    files = [f for f in os.listdir(selected_dir) if f.endswith('.wav')]
    if not files:
        return torch.zeros(1, 16000)
    path = os.path.join(selected_dir, random.choice(files))
    
    try:
        import librosa
        y_noise, _ = librosa.load(path, sr=16000, mono=True)
        if len(y_noise) >= 16000:
            start = random.randint(0, len(y_noise) - 16000)
            y_noise = y_noise[start:start+16000]
        else:
            y_noise = np.pad(y_noise, (0, 16000 - len(y_noise)), mode='constant')
        return torch.from_numpy(y_noise).unsqueeze(0)
    except Exception as e:
        print(f"Error loading noise {path}: {e}")
        return torch.zeros(1, 16000)

def mix_noise(clean_wf, noise_wf, snr_db=10):
    """Menggabungkan audio bersih dengan kebisingan berdasarkan skala SNR."""
    s_pwr = clean_wf.pow(2).mean()
    n_pwr = noise_wf.pow(2).mean()
    snr = 10 ** (snr_db / 10)
    scale = torch.sqrt(s_pwr / (snr * n_pwr + 1e-7))
    
    return clean_wf + (scale * noise_wf)

print("✅ Modul pengolah sinyal audio siap digunakan.")

### 5. Fungsi Ekstraksi Fitur Akustik (MFCC & CWT Scalogram)
Fungsi modular untuk mengekstrak 13 koefisien MFCC dan gambar visual Continuous Wavelet Transform (CWT) dengan wavelet induk complex morlet (`cmor1.5-1.0`). Dilengkapi pembersih memori ketat.

In [ ]:
def extract_features_stable(waveform, filename, label, suffix, output_dir, device):
    """Ekstraksi fitur ganda: MFCC (.pt) dan CWT Scalogram Gambar 2D (.png) pada 16.0 kHz."""
    # 1. MFCC (GPU)
    waveform_gpu = waveform.to(device)
    mfcc_tf = T.MFCC(sample_rate=16000, n_mfcc=13).to(device)
    mfcc = mfcc_tf(waveform_gpu)
    
    # Save MFCC tensor to PT file
    mfcc_dir = output_dir.replace("RespiScan_Train_Ready", "RespiScan_MFCC_Only")
    os.makedirs(os.path.join(mfcc_dir, str(label)), exist_ok=True)
    torch.save(mfcc.cpu(), os.path.join(mfcc_dir, f"{filename}{suffix}_mfcc.pt"))
    
    # 2. Scalogram CWT (CPU)
    y = waveform.numpy().flatten()
    widths = np.geomspace(2, 186, num=64)
    cwtmatr, _ = pywt.cwt(y, widths, 'cmor1.5-1.0')
    
    # Simpan sebagai Gambar visual secara langsung (cepat)
    lbl_folder = os.path.join(output_dir, str(label))
    os.makedirs(lbl_folder, exist_ok=True)
    
    matplotlib.image.imsave(os.path.join(lbl_folder, f"{filename}{suffix}.png"), np.abs(cwtmatr), cmap='jet')
    
    del waveform_gpu, mfcc, cwtmatr, y
    gc.collect()

print("✅ Modul ekstraktor fitur akustik dual-domain siap digunakan.")

### 6. Subject-Level Dataset Splitting, Class-Balanced Noise Mixing, & Augmentasi Akustik Sehat

In [ ]:
# Subject-Level Dataset Splitting, Class-Balanced Noise Mixing & Healthy Augmentation Protocol
import os
import random
import shutil
import pandas as pd
import json
import numpy as np
import torch
import torchaudio
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("=== MEMULAI SINKRONISASI PREPROCESSING BEBAS LEAKAGE ===")

# 1. Splitting tingkat subjek
tb_subjs = df_available[df_available['Label'] == 'TB']['subject'].tolist()
ntb_subjs = df_available[df_available['Label'] == 'NTB']['subject'].tolist()

tb_train, tb_temp = train_test_split(tb_subjs, test_size=0.30, random_state=42)
tb_val, tb_test = train_test_split(tb_temp, test_size=0.50, random_state=42)

ntb_train, ntb_temp = train_test_split(ntb_subjs, test_size=0.30, random_state=42)
ntb_val, ntb_test = train_test_split(ntb_temp, test_size=0.50, random_state=42)

train_subjects_set = set(tb_train + ntb_train)
val_subjects_set = set(tb_val + ntb_val)
test_subjects_set = set(tb_test + ntb_test)

print(f"Train: {len(train_subjects_set)} | Val: {len(val_subjects_set)} | Test: {len(test_subjects_set)}")

# 2. Persiapan Folder Output
DIR_OUTPUT_CWT = r"RespiScan_Train_Ready"
DIR_OUTPUT_MFCC = r"RespiScan_MFCC_Only"

for d in [DIR_OUTPUT_CWT, DIR_OUTPUT_MFCC]:
    if os.path.exists(d):
        try:
            shutil.rmtree(d)
        except Exception as e:
            print(f"Warning: Could not remove {d}: {e}")
    os.makedirs(d, exist_ok=True)
    for split in ['train', 'val', 'test']:
        for cls in ['Healthy', 'TB']:
            os.makedirs(os.path.join(d, split, cls), exist_ok=True)

# 3. Load noise files
noise_waveforms = []
for nd in ["DKITCHEN", "NPARK", "OHALLWAY"]:
    if os.path.exists(nd):
        for f in os.listdir(nd):
            if f.lower().endswith('.wav'):
                path = os.path.join(nd, f)
                try:
                    import librosa
                    y_noise, _ = librosa.load(path, sr=16000, mono=True)
                    if len(y_noise) >= 16000:
                        start = random.randint(0, len(y_noise) - 16000)
                        y_noise = y_noise[start:start+16000]
                    else:
                        y_noise = np.pad(y_noise, (0, 16000 - len(y_noise)), mode='constant')
                    noise_waveforms.append(torch.from_numpy(y_noise).unsqueeze(0))
                except Exception as e:
                    print(f"Error loading noise {path}: {e}")
if len(noise_waveforms) == 0:
    noise_waveforms.append(torch.zeros(1, 16000))

# 4. Loop Ekstraksi & Capping
MAX_SAMPLES_PER_SUBJ = 10
df_records = []

import librosa

def speed_perturb(waveform, speed_factor):
    y = waveform.squeeze(0).numpy()
    y_stretch = librosa.effects.time_stretch(y, rate=speed_factor)
    if len(y_stretch) > 16000:
        y_stretch = y_stretch[:16000]
    else:
        y_stretch = np.pad(y_stretch, (0, 16000 - len(y_stretch)), mode='constant')
    return torch.from_numpy(y_stretch).unsqueeze(0)

for idx, row in tqdm(df_available.iterrows(), total=len(df_available)):
    subj_name = row['subject']
    clean_subj = row['clean_subject']
    label = row['Label']
    class_folder = 'TB' if label == 'TB' else 'Healthy'
    
    if subj_name in train_subjects_set:
        split = 'train'
    elif subj_name in val_subjects_set:
        split = 'val'
    else:
        split = 'test'
        
    subj_wavs = subject_to_files.get(clean_subj, [])
    random.seed(42)  # For reproducible capping
    random.shuffle(subj_wavs)
    selected_wavs = subj_wavs[:MAX_SAMPLES_PER_SUBJ]
    
    for f in selected_wavs:
        w_path = os.path.join("dataset_tbc", f)
        base_name = f.replace('.wav', '')
        
        try:
            # Load & standardize audio
            raw_wf, raw_sr = torchaudio.load(w_path)
            clean_wf = standardize_audio(raw_wf, raw_sr, 16000, 16000)
            
            # Extract clean
            cwt_path = os.path.join(DIR_OUTPUT_CWT, split)
            extract_features_stable(clean_wf, base_name, class_folder, "_clean", cwt_path, DEVICE)
            df_records.append({'split': split, 'class': class_folder, 'subject_id': subj_name, 'filename': f"{base_name}_clean.png", 'augmentation': 'Clean'})
            
            # Augment only train
            if split == 'train':
                # Noisy augment
                noise_wf = random.choice(noise_waveforms)
                noisy_wf = mix_noise(clean_wf, noise_wf, snr_db=10)
                extract_features_stable(noisy_wf, base_name, class_folder, "_noisy", cwt_path, DEVICE)
                df_records.append({'split': split, 'class': class_folder, 'subject_id': subj_name, 'filename': f"{base_name}_noisy.png", 'augmentation': 'Noisy'})
                
                # Speed augment (only Healthy)
                if class_folder == 'Healthy':
                    for speed in [0.9, 1.1]:
                        s_wf = speed_perturb(clean_wf, speed)
                        s_label = 'slow' if speed == 0.9 else 'fast'
                        extract_features_stable(s_wf, base_name, class_folder, f"_{s_label}", cwt_path, DEVICE)
                        df_records.append({'split': split, 'class': class_folder, 'subject_id': subj_name, 'filename': f"{base_name}_{s_label}.png", 'augmentation': f"TimeStretch_{s_label.capitalize()}"})
                        
        except Exception as err:
            print(f"Error processing {f}: {err}")

# Save audit files
df_audit = pd.DataFrame(df_records)
df_audit.groupby(['split', 'class', 'augmentation']).size().reset_index(name='count').to_csv('augmentation_audit.csv', index=False)

summary_records = []
for s in ['train', 'val', 'test']:
    s_df = df_audit[df_audit['split'] == s]
    if s in ['val', 'test']:
        s_df_clean = s_df[s_df['augmentation'] == 'Clean']
    else:
        s_df_clean = s_df
        
    summary_records.append({
        'Split': s.upper(),
        'Total Subjects': s_df_clean['subject_id'].nunique(),
        'TB Subjects': s_df_clean[s_df_clean['class'] == 'TB']['subject_id'].nunique(),
        'Healthy Subjects': s_df_clean[s_df_clean['class'] == 'Healthy']['subject_id'].nunique(),
        'Total Samples': len(s_df_clean),
        'TB Samples': len(s_df_clean[s_df_clean['class'] == 'TB']),
        'Healthy Samples': len(s_df_clean[s_df_clean['class'] == 'Healthy'])
    })
pd.DataFrame(summary_records).to_csv('dataset_split_summary.csv', index=False)

pd.DataFrame([
    {'Check': 'Train vs Val Leakage', 'Overlap Count': len(train_subjects_set.intersection(val_subjects_set)), 'Overlap Subjects': list(train_subjects_set.intersection(val_subjects_set))},
    {'Check': 'Train vs Test Leakage', 'Overlap Count': len(train_subjects_set.intersection(test_subjects_set)), 'Overlap Subjects': list(train_subjects_set.intersection(test_subjects_set))},
    {'Check': 'Val vs Test Leakage', 'Overlap Count': len(val_subjects_set.intersection(test_subjects_set)), 'Overlap Subjects': list(val_subjects_set.intersection(test_subjects_set))}
]).to_csv('subject_leakage_check.csv', index=False)

config = {
    'dataset_source': 'TBscreen clinical cough database',
    'audio_processing': {
        'sampling_rate_hz': 16000,
        'duration_seconds': 1.0,
        'channels': 'mono',
        'cwt_wavelet': 'cmor1.5-1.0',
        'cwt_scales': 64
    },
    'train_augmentations': {
        'noise_mixing': 'Class-balanced (10 dB SNR mixed with DKITCHEN, NPARK, OHALLWAY)',
        'acoustic_perturbation': 'Healthy class time stretch'
    },
    'validation_and_testing': {
        'evaluation_mode': 'Clean-only (unaugmented)',
        'patient_level_aggregation': 'Majority voting / thresholding'
    }
}
with open('preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("\n📁 Compliance files berhasil disimpan:")
print(pd.read_csv('dataset_split_summary.csv'))